 # Cell 1: Pipeline Initialization

In [18]:
import os
import sys
import yaml
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

# 1. Dynamically locate the absolute project root folder
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, ".."))
else:
    PROJECT_ROOT = current_dir

print(f"🌍 Verified Project Root Anchor: {PROJECT_ROOT}")

# 2. Force load configuration
config_file_path = os.path.join(PROJECT_ROOT, "config.yaml")
with open(config_file_path, "r") as f:
    config = yaml.safe_load(f)

print("📝 Central config.yaml successfully mapped.")

🌍 Verified Project Root Anchor: C:\Users\DELL\python_projects\Intrusion-Detection-System-IDS-\intrusion_detection_system
📝 Central config.yaml successfully mapped.


# Cell 2: Running the Unified Preprocessing Routine

In [20]:
def read_raw_data(file_path, config):
    """Loads the raw NSL-KDD dataset and structurally maps headers."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing raw data file target at: {file_path}")
    df = pd.read_csv(file_path, header=None, names=config["data"]["columns"])
    if "difficulty_score" in df.columns:
        df = df.drop(columns=["difficulty_score"])
    return df

# Cell 3: Map Absolute Resource Paths & Run Preprocessing

In [21]:
# 1. Establish absolute file targets for raw assets
absolute_train_raw = os.path.join(PROJECT_ROOT, config["data"]["raw_train_path"])
absolute_test_raw = os.path.join(PROJECT_ROOT, config["data"]["raw_test_path"])

print("🚀 Reading raw data inputs directly via explicit system mapping...")
train_df = read_raw_data(absolute_train_raw, config)
test_df = read_raw_data(absolute_test_raw, config)

# 2. Binary classification target mapping
train_df["target"] = train_df["label"].apply(lambda x: 0 if x == "normal" else 1)
test_df["target"] = test_df["label"].apply(lambda x: 0 if x == "normal" else 1)

train_df = train_df.drop(columns=["label"])
test_df = test_df.drop(columns=["label"])

# 3. Categorical processing (One-Hot Encoding)
categorical_cols = ["protocol_type", "service", "flag"]
combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
combined = pd.get_dummies(combined, columns=categorical_cols, drop_first=True)

train_processed = combined.iloc[:len(train_df)].copy()
test_processed = combined.iloc[len(train_df):].copy()

X_train = train_processed.drop(columns=["target"])
X_test = test_processed.drop(columns=["target"])

# 4. Standard Scaling operations
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
train_scaled["target"] = train_processed["target"].values

test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
test_scaled["target"] = test_processed["target"].values

# 5. FORCE ABSOLUTE DIRECTORIES & SAVE ARTIFACTS
processed_directory = os.path.join(PROJECT_ROOT, "data", "processed")
models_directory = os.path.join(PROJECT_ROOT, "models")

os.makedirs(processed_directory, exist_ok=True)
os.makedirs(models_directory, exist_ok=True)

# Generate exact output path instances
output_train_csv = os.path.join(processed_directory, "train_processed.csv")
output_test_csv = os.path.join(processed_directory, "test_processed.csv")
output_scaler_pkl = os.path.join(models_directory, "scaler.pkl")

# Save everything using absolute paths
train_scaled.to_csv(output_train_csv, index=False)
test_scaled.to_csv(output_test_csv, index=False)
joblib.dump(scaler, output_scaler_pkl)

print("\n✨ Success! All data files transformed and written flawlessly.")

🚀 Reading raw data inputs directly via explicit system mapping...

✨ Success! All data files transformed and written flawlessly.


# Cell 4: Verify Generated Artifact Layouts (Data Integrity Check)

In [22]:
print("=== Post-Execution Verification ===")
print(f"Processed train data matrix:  {'✔️ Found' if os.path.exists(output_train_csv) else '❌ Missing'}")
print(f"Processed test data matrix:   {'✔️ Found' if os.path.exists(output_test_csv) else '❌ Missing'}")
print(f"Saved scaler binary asset:    {'✔️ Found' if os.path.exists(output_scaler_pkl) else '❌ Missing'}")

print("\n📋 Scaled Feature Matrix Sample Layout (First 3 Records):")
display(train_scaled.head(3))

=== Post-Execution Verification ===
Processed train data matrix:  ✔️ Found
Processed test data matrix:   ✔️ Found
Saved scaler binary asset:    ✔️ Found

📋 Scaled Feature Matrix Sample Layout (First 3 Records):


,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH,target
0,-0.110249,-0.007679,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,-0.809262,-0.011664,...,-0.11205,-0.028606,-0.139982,-0.618438,-0.053906,-0.031767,-0.019726,0.825150,-0.046432,0
1,-0.110249,-0.007737,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,-0.809262,-0.011664,...,-0.11205,-0.028606,-0.139982,-0.618438,-0.053906,-0.031767,-0.019726,0.825150,-0.046432,0
2,-0.110249,-0.007762,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,-0.027023,-0.809262,-0.011664,...,-0.11205,-0.028606,-0.139982,1.616978,-0.053906,-0.031767,-0.019726,-1.211901,-0.046432,1
